# 23CSE301 Machine Learning - Capstone Project
## Regression Track: BigMart Sales Prediction

This notebook covers the complete end-to-end Machine Learning pipeline for predicting **BigMart Item Sales** (`Item_Outlet_Sales`).

### Pipeline Steps:
1. **Exploratory Data Analysis (EDA)**: Data loading, summary statistics, missing values audit, and visual exploration.
2. **Data Cleaning & Preprocessing**: Handling missing values, categorical encoding, and feature scaling.
3. **Feature Engineering**: Creating domain-specific engineered features.
4. **Model Training & Comparison (First 5 Algorithms)**:
   - Linear Regression (Baseline)
   - Ridge Regression ($L_2$ Regularization)
   - Lasso Regression ($L_1$ Regularization)
   - ElasticNet Regression ($L_1 + L_2$ Regularization)
   - Polynomial Regression (`PolynomialFeatures` + Linear Regression)
5. **Model Evaluation & Visualization**: $R^2$, RMSE, MAE, 5-Fold Cross Validation $R^2$, Residual Plots, Predicted vs. Actual plots.
6. **Model Serialization**: Saving the best model pipeline via `joblib`.

---
## Step 1: Environment Setup & Library Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, GridSearchCV, KFold, cross_val_score
from sklearn.preprocessing import StandardScaler, LabelEncoder, OneHotEncoder
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.svm import SVR
from sklearn.neighbors import KNeighborsRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import warnings
warnings.filterwarnings('ignore')

---
## Step 2: Dataset Loading & Audit (Section A1)

In [ ]:
data_path = r'../data/regression/data.csv'
df = pd.read_csv(data_path)

print("Dataset Shape:", df.shape)
print("\nData Types:")
print(df.dtypes)
print("\nMissing Values:")
print(df.isnull().sum())
print("\nTarget Distribution Summary:")
print(df['Item_Outlet_Sales'].describe())

---
## Step 3: Exploratory Data Analysis (EDA) (Section A2 & A3)

---
## Step 4: Preprocessing & Feature Engineering (Section B)

In [ ]:
df['Item_Fat_Content'] = df['Item_Fat_Content'].replace({'LF': 'Low Fat', 'low fat': 'Low Fat', 'reg': 'Regular'})

# impute missing values

df['Item_Weight'] = df.groupby('Item_Identifier')['Item_Weight'].transform(lambda x: x.fillna(x.mean()))
df['Item_Weight'] = df['Item_Weight'].fillna(df['Item_Weight'].mean()) # Fallback for remaining

# Outlet_Size: impute with mode based on Outlet_Type
outlet_size_mode = df.groupby('Outlet_Type')['Outlet_Size'].apply(lambda x: x.mode()[0] if not x.mode().empty else 'Medium').to_dict()
df['Outlet_Size'] = df.apply(lambda x: outlet_size_mode[x['Outlet_Type']] if pd.isna(x['Outlet_Size']) else x['Outlet_Size'], axis=1)

# 3. Handle Item_Visibility 0 values
item_visibility_mean = df.groupby('Item_Identifier')['Item_Visibility'].mean()
df['Item_Visibility'] = df.apply(lambda x: item_visibility_mean[x['Item_Identifier']] if x['Item_Visibility'] == 0 else x['Item_Visibility'], axis=1)


# extract Item_Category
df['Item_Category'] = df['Item_Identifier'].apply(lambda x: x[:2])
df.loc[df['Item_Category'] == 'NC', 'Item_Fat_Content'] = 'Non-Edible'
df['Outlet_Age'] = 2013 - df['Outlet_Establishment_Year']

# split back to Train/Test before Encoding
train_df = df[df['source'] == 'train'].copy()
test_df = df[df['source'] == 'test'].copy()

drop_cols = ['Item_Identifier', 'Outlet_Identifier', 'Outlet_Establishment_Year', 'source']
train_df.drop(columns=drop_cols, inplace=True)
test_df.drop(columns=drop_cols + ['Item_Outlet_Sales'], inplace=True)

# Encoding
le = LabelEncoder()
train_df['Outlet_Size'] = le.fit_transform(train_df['Outlet_Size'])
train_df['Outlet_Location_Type'] = le.fit_transform(train_df['Outlet_Location_Type'])
test_df['Outlet_Size'] = le.transform(test_df['Outlet_Size'])
test_df['Outlet_Location_Type'] = le.transform(test_df['Outlet_Location_Type'])

categorical_cols = ['Item_Fat_Content', 'Item_Type', 'Outlet_Type', 'Item_Category']
train_df = pd.get_dummies(train_df, columns=categorical_cols, drop_first=True)
test_df = pd.get_dummies(test_df, columns=categorical_cols, drop_first=True)

train_cols = train_df.columns.drop('Item_Outlet_Sales')
for col in train_cols:
    if col not in test_df.columns:
        test_df[col] = 0
test_df = test_df[train_cols]

X = train_df.drop('Item_Outlet_Sales', axis=1)
y = train_df['Item_Outlet_Sales']

X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

# Scaling
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)
test_scaled = scaler.transform(test_df)

---
## Step 5: Regression Models (First 5 Algorithms) (Section C)

### Model 1: Linear Regression
### Model 2: Ridge Regression
### Model 3: Lasso Regression
### Model 4: ElasticNet Regression
### Model 5: Polynomial Regression

In [ ]:
# Model training, hyperparameter tuning via GridSearchCV, evaluation

---
## Step 6: Regression Models (First 5 Algorithms) (Section C)

### Model 6: Decision Tree Regressor
### Model 7: Random Forest Regressor
### Model 8: Gradient Boosting Regressor
### Model 9:  Support Vector Regressor (SVR)
### Model 10: K-Nearest Neighbors Regressor 

---
## Step 7: Model Evaluation Summary & Visualizations

In [ ]:
# Summary metrics dataframe, 5-fold CV, residual plot, predicted vs actual plot

---
## Step 8: Model Export

In [ ]:
# Save trained best model pipeline to models/regression_model.pkl